In [1]:
!pip install -q agent-framework

# Agent

In [2]:
import os
from dotenv import load_dotenv
from agent_framework.openai import OpenAIChatClient
load_dotenv()

agent = OpenAIChatClient(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openai/gpt-5-nano",   # any OpenRouter slug, provider/model
).as_agent(
    name="OpenRouterAgent",
    instructions="You are a helpful assistant.",
)

result = await agent.run("What's the weather in Bengaluru?")
print(result)

I don’t have live weather data here. I can help in two ways:

- Current conditions: I can guide you to check quickly on your device or tell you how to fetch it from a site (Google, Weather.com, IMD, etc.).
- Climate overview: I can give you typical Bengaluru weather by season.

If you’d like the current conditions, tell me if you want a quick lookup guide or if you want a forecast for the next few days. Or, here’s a brief climate snapshot for Bengaluru:

- Winters (Nov–Feb): generally mild. Day 15–28°C, nights cooler (around 15–18°C).
- Spring to early summer (Mar–May): warm, 20–34°C; May can be hot.
- Monsoon (Jun–Sep): 20–28°C with frequent rain and higher humidity.
- Post-monsoon (Oct–Nov): pleasant, around 18–29°C.
- Typical note: nights are cool in winter; humidity is higher during the monsoon.

Would you like me to help you find the current temperature or a short-term forecast? If you want, you can also tell me the exact time (today/tonight) and I’ll tailor it.


In [9]:
import os
from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient

@tool
def get_stock_price(ticker: str) -> str:
    """Get the latest price for an NSE ticker."""
    return f"{ticker}: ₹2,450.30"

agent = OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openai/gpt-5-nano",
).as_agent(
    name="StockAgent",
    instructions="You help with NSE stock queries.",
    tools=get_stock_price,
)

result = await agent.run("What's the price of RELIANCE?")
print(result)

RELIANCE is currently ₹2,450.30 on NSE.


# Harness

In [14]:
!pip install -q --upgrade --pre agent-framework-tools

The Harness shell isn't just another tool. It's a special integration where the framework expects the client to understand additional metadata and protocols for:

shell execution,
environment providers,
approval flows,
execution state.

The OpenAI-compatible clients only support standard chat completion and function/tool calling.

For using Shell, it needs to be built on Azure AI Foundry.

In [16]:
from agent_framework import create_harness_agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openai/gpt-5-nano",
)

agent = create_harness_agent(client)

from agent_framework_tools.shell import (
    LocalShellTool,
    ShellPolicy
)

async with LocalShellTool(
    workdir="./workspace",
    confine_workdir=True,
    policy=ShellPolicy(
        denylist=[
            r"\brm\s+-rf\b",
            r"\bsudo\b"
        ]
    )
) as shell:

    agent = create_harness_agent(
        client=client,
        shell_executor=shell
    )

Shell tool not available: client 'OpenAIChatCompletionClient' does not implement SupportsShellTool. Skipping the shell tool and environment provider.


# Workflow

In [2]:
import asyncio
from agent_framework import (
    Executor,
    Workflow,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    handler,
)
from typing_extensions import Never


# Example 1: A custom Executor subclass using introspection (traditional approach)
# ---------------------------------------------------------------------------------
#
# Subclassing Executor lets you define a named node with lifecycle hooks if needed.
# The work itself is implemented in an async method decorated with @handler.
#
# Handler signature contract:
# - First parameter is the typed input to this node (here: text: str)
# - Second parameter is a WorkflowContext[T_Out], where T_Out is the type of data this
#   node will emit via ctx.send_message (here: T_Out is str)
#
# Within a handler you typically:
# - Compute a result
# - Forward that result to downstream node(s) using ctx.send_message(result)
class UpperCase(Executor):
    def __init__(self, id: str):
        super().__init__(id=id)

    @handler
    async def to_upper_case(self, text: str, ctx: WorkflowContext[str]) -> None:
        """Convert the input to uppercase and forward it to the next node.

        Note: The WorkflowContext is parameterized with the type this handler will
        emit. Here WorkflowContext[str] means downstream nodes should expect str.
        """

        result = text.upper()

        # Send the result to the next executor in the workflow.
        await ctx.send_message(result)


# Example 2: A standalone function-based executor using introspection
# --------------------------------------------------------------------
#
# For simple steps you can skip subclassing and define an async function with the
# same signature pattern (typed input + WorkflowContext[T_Out, T_W_Out]) and decorate it with
# @executor. This creates a fully functional node that can be wired into a flow.


@executor(id="reverse_text_executor")
async def reverse_text(text: str, ctx: WorkflowContext[Never, str]) -> None:
    """Reverse the input string and yield the workflow output.

    This node yields the final output using ctx.yield_output(result).
    The workflow will complete when it becomes idle (no more work to do).

    The WorkflowContext is parameterized with two types:
    - T_Out = Never: this node does not send messages to downstream nodes.
    - T_W_Out = str: this node yields workflow output of type str.
    """
    result = text[::-1]

    # Yield the output - the workflow will complete when idle
    await ctx.yield_output(result)


# Example 3: Using explicit type parameters on @handler
# -----------------------------------------------------
#
# Instead of relying on type introspection, you can explicitly specify input,
# output, and/or workflow_output on the @handler decorator. This is "all or nothing":
# when ANY explicit parameter is provided, ALL types come from explicit parameters
# (introspection is completely disabled). The input parameter is required.
#
# This is useful when:
# - You want to accept multiple types (union types) without complex type annotations
# - The function signature uses Any or a base type for flexibility
# - You want to decouple the runtime type routing from the static type annotations


class ExclamationAdder(Executor):
    """An executor that adds exclamation marks, demonstrating explicit @handler types.

    This example shows how to use explicit input and output parameters
    on the @handler decorator instead of relying on introspection from the function
    signature. This approach is especially useful for union types.
    """

    def __init__(self, id: str):
        super().__init__(id=id)

    @handler(input=str, output=str)
    async def add_exclamation(self, message, ctx) -> None:  # type: ignore
        """Add exclamation marks to the input.

        Note: The input=str and output=str are explicitly specified on @handler,
        so the framework uses those instead of introspecting the function signature.
        The WorkflowContext here has no type parameters because the explicit types
        on @handler take precedence.
        """
        result = f"{message}!!!"
        await ctx.send_message(result)  # type: ignore


# 1
def create_workflow() -> Workflow:
    """Create a fresh workflow with isolated state.

    Wrapping workflow construction in a helper function ensures each call
    produces independent executor instances. This is the recommended pattern
    for reuse — call create_workflow() each time you need a new workflow so
    that no state leaks between runs.
    """
    upper_case = UpperCase(id="upper_case_executor")

    return WorkflowBuilder(start_executor=upper_case).add_edge(upper_case, reverse_text).build()


async def main():
    """Build and run workflows using the fluent builder API."""

    # Workflow 1: Using the helper function pattern for state isolation
    # ------------------------------------------------------------------
    # Each call to create_workflow() returns a workflow with fresh executor
    # instances. This is the recommended pattern when you need to run the
    # same workflow topology multiple times with clean state.
    workflow1 = create_workflow()

    # Run the workflow by sending the initial message to the start node.
    # The run(...) call returns an event collection; its get_outputs() method
    # retrieves the outputs yielded by any terminal nodes.
    print("Workflow 1 (introspection-based types):")
    events1 = await workflow1.run("hello world")
    print(events1.get_outputs())
    print("Final state:", events1.get_final_state())

    # Workflow 2: Using explicit type parameters on @handler
    # -------------------------------------------------------
    upper_case = UpperCase(id="upper_case_executor")
    exclamation_adder = ExclamationAdder(id="exclamation_adder")

    # This workflow demonstrates the explicit input/output feature:
    # exclamation_adder uses @handler(input=str, output=str) to
    # explicitly declare types instead of relying on introspection.
    workflow2 = (
        WorkflowBuilder(start_executor=upper_case)
        .add_edge(upper_case, exclamation_adder)
        .add_edge(exclamation_adder, reverse_text)
        .build()
    )

    print("\nWorkflow 2 (explicit @handler types):")
    events2 = await workflow2.run("hello world")
    print(events2.get_outputs())
    print("Final state:", events2.get_final_state())

    """
    Sample Output:

    Workflow 1 (introspection-based types):
    ['DLROW OLLEH']
    Final state: WorkflowRunState.IDLE

    Workflow 2 (explicit @handler types):
    ['!!!DLROW OLLEH']
    Final state: WorkflowRunState.IDLE
    """


await main()

Workflow 1 (introspection-based types):
['DLROW OLLEH']
Final state: WorkflowRunState.IDLE

Workflow 2 (explicit @handler types):
['!!!DLROW OLLEH']
Final state: WorkflowRunState.IDLE


/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_64878/3584970791.py:119: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  return WorkflowBuilder(start_executor=upper_case).add_edge(upper_case, reverse_text).build()
/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_64878/3584970791.py:152: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()


In [6]:
import asyncio
import os
from typing import cast
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv

from agent_framework import Agent, AgentResponse, WorkflowBuilder, InMemoryCheckpointStorage
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

# Load environment variables from .env file
load_dotenv()

"""
Step 2: Agents in a Workflow non-streaming

This sample creates two agents: a Writer agent creates or edits content, and a Reviewer agent which
evaluates and provides feedback.

Purpose:
Show how to create agents from FoundryChatClient and use them directly in a workflow. Demonstrate
how agents can be used in a workflow.

Prerequisites:
- FOUNDRY_PROJECT_ENDPOINT must be your Azure AI Foundry Agent Service (V2) project endpoint.
- FOUNDRY_MODEL must be the deployment name of a model in your Foundry project.
- Authentication via azure-identity. Use AzureCliCredential and run az login before executing the sample.
- Basic familiarity with WorkflowBuilder, edges, events, and streaming or non-streaming runs.
"""


async def main():
    """Build and run a simple two node agent workflow: Writer then Reviewer."""
    # Create the Azure chat client. AzureCliCredential uses your current az login.
    client = OpenAIChatClient(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        model="openai/gpt-5-nano",   # any OpenRouter slug, provider/model
    )
    writer_agent = Agent(
        client=client,
        instructions=(
            "You are an excellent content writer. You create new content and edit contents based on the feedback."
        ),
        name="writer",
    )

    reviewer_agent = Agent(
        client=client,
        instructions=(
            "You are an excellent content reviewer."
            "Provide actionable feedback to the writer about the provided content."
            "Provide the feedback in the most concise manner possible."
        ),
        name="reviewer",
    )

    # Build the workflow using the fluent builder.
    # Set the start node via constructor and connect an edge from writer to reviewer.
    workflow = WorkflowBuilder(start_executor=writer_agent).add_edge(writer_agent, reviewer_agent).build()

    # Run the workflow with the user's initial message.
    # For foundational clarity, use run (non streaming) and print the terminal event.
    events = await workflow.run(
        message="Create a slogan for a new electric SUV that is affordable and fun to drive.",
        checkpoint_storage=InMemoryCheckpointStorage())

    outputs = events.get_outputs()
    # The outputs of the workflow are whatever the agents produce. So the outputs are expected to be a list
    # of `AgentResponse` from the agents in the workflow.
    outputs = cast(list[AgentResponse], outputs)
    for output in outputs:
        print(f"{output.messages[0].author_name}: {output.text}\n")

    # Summarize the final run state (e.g., COMPLETED)
    print("Final state:", events.get_final_state())
    return workflow

workflow = await main()

/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_64878/3758560856.py:60: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  workflow = WorkflowBuilder(start_executor=writer_agent).add_edge(writer_agent, reviewer_agent).build()


writer: Here are several slogan options in different tones. Pick the vibe you like, or tell me your target audience and tone and I’ll tailor more.

- Electric fun, prices you’ll love.
- Drive electric. Have a blast. Save big.
- Affordably electric. Unstoppably fun.
- Plug in. Power up. Wallet happy.
- Zero emissions. Maximum fun. Minimum cost.
- Drive electric. Smile more. Spend less.
- Charge the fun. Shrink the bill.
- Electric thrills, everyday affordability.

reviewer: Actionable feedback:

- What works
  - Clear focus on affordability and fun.
  - Short, punchy lines good for slogans and social media.
  - Mix of tones (playful, value-driven) gives options.

- What to improve
  - Add explicit “SUV” to avoid ambiguity.
  - Tighten for logo/tagline use (keep to 3–5 words when possible).
  - Watch clichés (Plug in, Zero emissions) and aim for more distinctive phrasing.
  - Align tone across options for a cohesive brand voice; pick 1–2 tones to scale.

- Quick rewrites (explicitly SUV,

In [7]:
workflow.id

'f01aeec8-dae7-451b-80fc-4bd4804dace8'

In [9]:
workflow.to_json()

'{"name": "WorkflowBuilder-767f58bb-c1e1-4fe1-9d08-b3c2549bdba2", "id": "f01aeec8-dae7-451b-80fc-4bd4804dace8", "start_executor_id": "writer", "max_iterations": 100, "edge_groups": [{"id": "InternalEdgeGroup/2989658d-31bf-4c60-994e-2dfbb78c867c", "type": "InternalEdgeGroup", "edges": [{"source_id": "internal:writer", "target_id": "writer"}]}, {"id": "InternalEdgeGroup/cbb64df6-94b6-4059-8dd2-a93823282e40", "type": "InternalEdgeGroup", "edges": [{"source_id": "internal:reviewer", "target_id": "reviewer"}]}, {"id": "SingleEdgeGroup/682d426e-406a-4063-8d3a-5551bf0551e4", "type": "SingleEdgeGroup", "edges": [{"source_id": "writer", "target_id": "reviewer"}]}], "executors": {"writer": {"id": "writer", "type": "AgentExecutor"}, "reviewer": {"id": "reviewer", "type": "AgentExecutor"}}, "output_executors": null, "intermediate_executors": null}'

In [10]:
events = await workflow.run("Target explicitly for SUV")

outputs = events.get_outputs()

outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

writer: Could you clarify what you mean by “Target explicitly for SUV”? A few possibilities:

- Targeting an advertising audience explicitly for SUV buyers
- Labeling data/products as SUV in a dataset or taxonomy
- Aiming product messaging specifically at SUV owners/users

If you’re asking about advertising, here’s a concise starter you can use. You can adapt per platform (Facebook/Meta, Google Ads, etc.) and your goal (awareness, consideration, conversions).

What to target (audience ideas)
- In-market/intent:
  - In-market for SUVs or crossovers
  - Auto shoppers researching 3-row SUVs, family SUVs, AWD SUVs
- Demographics:
  - Age 30–54
  - Household income mid-to-high (e.g., $75k+)
  - Homeowners, suburban/exurban residents
- Interests and behaviors:
  - Family travel, road trips, camping, outdoor activities
  - Safety-conscious buyers, car-seat and family-friendly features
- Geography:
  - Suburban areas with higher SUV ownership; regional preferences (vary by country/region)

Pla

# Thread isolation

LangGraph: You manage conversation threads implicitly by passing a user-defined thread_id inside a configuration dictionary (config={"configurable": {"thread_id": "xyz"}}). LangGraph automatically looks up the latest checkpoint associated with that thread ID.

Microsoft Agent Framework: State isolation and resumption are managed explicitly via CheckpointStorage engines and unique, framework-generated checkpoint_id strings. To resume a specific execution thread, you pass the checkpoint_id directly into the workflow's run() method.

In [39]:
import os
from dotenv import load_dotenv
from agent_framework.openai import OpenAIChatClient
load_dotenv()

agent = OpenAIChatClient(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openai/gpt-5-nano",   # any OpenRouter slug, provider/model
).as_agent(
    name="OpenRouterAgent",
    instructions="You are a helpful assistant.",
)

async def main() -> None:

    # <multi_turn>
    # Create a session to maintain conversation history
    session = agent.create_session(session_id="124125wt")

    # First turn
    result = await agent.run("My name is Alice and I love hiking.", session=session)
    print(f"Agent: {result}\n")

    # Second turn — the agent should remember the user's name and hobby
    result = await agent.run("What do you remember about me?", session=session)
    print(f"Agent: {result}")
    # </multi_turn>

await main()

Agent: Hi Alice! Nice to meet a fellow hiker. How can I help today? Some ideas:
- Recommend trails near you (tell me your location and preferred distance/elevation)
- Create a packing list for a day hike or a weekend trip
- Share safety tips and gear basics
- Build a beginner-friendly training plan to boost endurance
- Plan a custom hiking itinerary for a trip

What would you like to start with? If you want, I can also just give you a quick day-hike packing checklist right now.

Agent: I don’t have memory of you from past chats. In this conversation, I can remember details you share so I can help better, but I don’t retain information after we’re done unless your app or platform provides a memory feature.

If you want, tell me what you’d like me to remember for this chat (for example: your name, your goals, topics you’re interested in, or preferred style). I can use that to tailor my responses now.

Would you like me to remember anything for this conversation, or enable memory across s